# Real or AI-generated? — CPS843 term project
Runs the whole project on **Google Colab**: download data → train 3 detectors → run the experiments → make the report's tables and figures → live demo.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`, then run the cells top to bottom (click ▶ on each, wait for it to finish).
The code runs on Colab's fast local disk; models and results are backed up to your Google Drive after every big step.

## 1. Connect Google Drive (backup for models and results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/cps843-ai-image-detector'   # backup folder in your Drive
os.makedirs(PROJECT, exist_ok=True)

def save():
    # copy models + results to Drive so a disconnect loses nothing
    os.system(f'mkdir -p "{PROJECT}/outputs" && cp -r models results "{PROJECT}/outputs/"')
    print('backed up to Drive:', PROJECT + '/outputs')

## 2. Get the code
Clones the group's public GitHub repo.
Anything backed up earlier in Drive (`outputs/`) is restored automatically.

In [ ]:
REPO_URL = 'https://github.com/ridham5/ai-image-detector.git'
%cd /content
if not os.path.exists('/content/ai-image-detector'):
    !git clone -q {REPO_URL} /content/ai-image-detector
%cd /content/ai-image-detector
!git pull -q
# restore earlier models/results from Drive, if any
!cp -r "{PROJECT}/outputs/models" "{PROJECT}/outputs/results" . 2>/dev/null; true
!ls

## 3. Install the extra libraries (PyTorch is already on Colab)

In [ ]:
!pip -q install open_clip_torch kagglehub gradio
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to T4 GPU')

## 4. Download CIFAKE
Saved on Colab's local disk. If this cell fails with a login / 401 / 403 error, run the **optional** cell below it, then run this cell again.

In [ ]:
os.environ['KAGGLEHUB_CACHE'] = '/content/kaggle_cache'
!python scripts/download_cifake.py
!ls data/cifake/train data/cifake/test

**Optional — only if the download asked for a login.** On kaggle.com: sign in → profile picture → *Settings* → *API* → *Create New Token*.
A file `kaggle.json` downloads. Run this cell and choose that file.

In [ ]:
from google.colab import files
files.upload()   # pick kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print('done - now run the download cell again')

## 5. Settings
**First run:** keep these small to check everything works (~20–30 min total).
**Final run:** change to `N_TRAIN = 20000`, `N_TEST = 5000`, `EPOCHS = 3`, then re-run from cell 7.

In [ ]:
N_TRAIN = 2000    # training images per class (CIFAKE has 50,000 per class)
N_TEST  = 500     # test images per class      (CIFAKE has 10,000 per class)
EPOCHS  = 1       # ResNet-50 passes over the training data

## 6. Dataset figures (sample images + Fourier spectra) — no training needed

In [ ]:
!python scripts/figures.py --data-dir data/cifake/test --n-spectrum 2000
from IPython.display import Image, display, Markdown
for f in ['fig_samples.png', 'fig_spectrum.png', 'fig_radial.png']:
    display(Image(f'results/{f}', width=700))
save()

## 7. Train Method 1 — FFT features + SVM (CPU)

In [ ]:
!python scripts/train.py --method fft_svm --train-dir data/cifake/train --max-per-class {min(N_TRAIN, 10000)}
save()

## 8. Train Method 2 — ResNet-50 fine-tuning (GPU; the longest step)

In [ ]:
!python scripts/train.py --method resnet50 --train-dir data/cifake/train --max-per-class {N_TRAIN} --epochs {EPOCHS} --batch-size 64
save()

## 9. Train Method 3 — CLIP + linear probe (GPU)

In [ ]:
!python scripts/train.py --method clip_probe --train-dir data/cifake/train --max-per-class {N_TRAIN}
save()

## 10. Experiments 1 + 2 — main results and robustness
Each method is tested on clean images and 7 degraded versions (JPEG, blur, resize, noise). Every run adds a row to `results/results.csv`.

In [ ]:
for m in ['fft_svm', 'resnet50', 'clip_probe']:
    !python scripts/evaluate.py --method {m} --test-dir data/cifake/test --test-name cifake --max-per-class {N_TEST} --robustness
save()

## 11. Experiment 3 — generalisation to other generators (optional)
Needs GenImage subsets (see README → *GenImage*) arranged as `data/genimage/<generator>/val/ai/...` and `.../val/nature/...`.
If that folder doesn't exist, this cell does nothing.

In [ ]:
import glob
for g in sorted(glob.glob('data/genimage/*/')):
    name = os.path.basename(g.rstrip('/'))
    for m in ['fft_svm', 'resnet50', 'clip_probe']:
        !python scripts/evaluate.py --method {m} --test-dir {g}val --test-name {name} --max-per-class 2000
save()

## 12. Tables and figures for the report

In [ ]:
!python scripts/make_tables.py --main-test cifake
for t in ['table_main', 'table_robustness', 'table_generalization']:
    if os.path.exists(f'results/{t}.md'):
        display(Markdown(open(f'results/{t}.md').read()))
for f in ['fig_robustness.png', 'fig_generalization.png']:
    if os.path.exists(f'results/{f}'):
        display(Image(f'results/{f}', width=700))
save()

## 13. Grad-CAM — where does ResNet-50 look?

In [ ]:
!python scripts/gradcam_figure.py --test-dir data/cifake/test --n 3
display(Image('results/fig_gradcam.png', width=700))
save()

## 14. Live demo
Click the public link it prints (valid ~72 h). Stop the cell (■) to end the demo.

In [ ]:
!python app.py --share